In [1]:
# Imports, env vars, logging/warning suppression

%env MUJOCO_GL=egl
# Pin to GPUs 1,2 (GPU 0 is held by another user; GPU 3 omitted to keep
# NUM_DEVICES a power-of-two divisor of NUM_ENVS=1024 and BATCH_SIZE=256).
# Must be set before `import jax`. The training config below assumes 2 devices.
%env CUDA_VISIBLE_DEVICES=0,1,2,3
import datetime
import functools
import os
import time
import warnings

import flax.linen as linen
import jax
import jax.numpy as jp
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np
import warp as wp
from absl import logging
from brax.io import model
from brax.training.acme import running_statistics
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from etils import epath
from mujoco_playground import registry, wrapper
from mujoco_playground._src.wrapper import Wrapper as MPGWrapper

import twmr  # noqa: F401 — registers TWMRLegFlat / TWMRLegTerr
from twmr.networks import STUDENT_OBS_SIZE, TEACHER_OBS_SIZE

wp.config.quiet = True
xla_flags = os.environ.get("XLA_FLAGS", "")
xla_flags += " --xla_gpu_triton_gemm_any=True"
os.environ["XLA_FLAGS"] = xla_flags
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
logging.set_verbosity(logging.WARNING)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

NUM_DEVICES = jax.local_device_count()

media.set_ffmpeg("/home/imeg2025/Transformable-Leg-Wheel-Robot/.pixi/envs/default/bin/ffmpeg")

print("jax:", jax.__version__, " |  device:", jax.default_backend(), " |  num_devices:", NUM_DEVICES)


env: MUJOCO_GL=egl
env: CUDA_VISIBLE_DEVICES=0,1,2,3
jax: 0.6.2  |  device: gpu  |  num_devices: 4


In [2]:
# ── Training config ───────────────────────────────────────────────────────────
# Plain-PPO baseline: same env, DR, and hyperparameters as phase1_run, but
# the policy sees only the 29-dim student obs (no privileged info, no encoder,
# no latent z). Trained model saved under logs/<ENV_NAME>-PPO-<ts>/ so
# phase2_run can auto-discover it for the 5th-condition comparison.
ENV_NAME             = "TWMRLegTerr"  # match phase2_run's ENV_NAME
SEED                 = 1

NUM_TIMESTEPS        = 3_000_000
EPISODE_LENGTH       = 500        # 5 s at ctrl_dt = 0.02 s/step
# Config assumes NUM_DEVICES == 2 (pinned via CUDA_VISIBLE_DEVICES in cell 0).
# Brax PPO has three divisibility constraints — all satisfied here:
#   1. NUM_ENVS % NUM_DEVICES == 0                  → 1024 % 2 == 0 ✓
#   2. BATCH_SIZE * NUM_MINIBATCHES % NUM_ENVS == 0 → 2048 % 1024 == 0 ✓
#   3. BATCH_SIZE % NUM_DEVICES == 0                → 256 % 2 == 0 ✓
NUM_ENVS             = 1024
NUM_EVAL_ENVS        = 128
NUM_EVALS            = 20         # evals every 5 % → hits 5, 10, …, 100 %
UNROLL_LENGTH        = 10
BATCH_SIZE           = 256
NUM_MINIBATCHES      = 8
NUM_UPDATES_PER_BATCH = 8

LEARNING_RATE        = 5e-4
ENTROPY_COST         = 5e-3
DISCOUNTING          = 0.97
REWARD_SCALING       = 1.0
CLIPPING_EPSILON     = 0.3
MAX_GRAD_NORM        = 1.0
POLICY_HIDDEN        = (64, 64, 64)
VALUE_HIDDEN         = (64, 64, 64)
LOGDIR               = "logs"
# Render a video at these training percentages.
TARGET_PCTS          = set({5, 20, 50, 100})
# ─────────────────────────────────────────────────────────────────────────────


In [3]:
# ── Logging directory ────────────────────────────────────────────────────────
# The "-PPO-" tag is what phase2_run's auto-discovery glob looks for. Do not
# rename the prefix without updating phase2_run cell 5.
exp_name = f"{ENV_NAME}-PPO-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
logdir   = epath.Path(LOGDIR).resolve() / exp_name
logdir.mkdir(parents=True, exist_ok=True)
ckpt_dir = logdir / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
print(f"Logs: {logdir}")


# ── Obs-slicing wrapper ──────────────────────────────────────────────────────
# Raw TWMRLeg* env returns 38-dim teacher obs = [student(29) | priv(9)].
# Plain-PPO baseline must see only the 29-dim student slice. We wrap at the
# outermost layer so DR / EpisodeWrapper / AutoResetWrapper still see the full
# 38-dim obs internally; the slice is applied only on the way out.
class StudentObsWrapper(MPGWrapper):
    """Slice state.obs to the first STUDENT_OBS_SIZE dims on reset and step.

    The inner env rebuilds state.obs from state.data, so its step ignores the
    incoming obs values — but EpisodeWrappers lax.scan still type-checks the
    carry. We therefore pad the sliced obs back to TEACHER_OBS_SIZE before
    handing it to the inner step, then re-slice the result.
    """

    def reset(self, rng):
        s = self.env.reset(rng)
        return s.replace(obs=s.obs[..., :STUDENT_OBS_SIZE])

    def step(self, state, action):
        pad_width = TEACHER_OBS_SIZE - STUDENT_OBS_SIZE
        pad = jp.zeros(state.obs.shape[:-1] + (pad_width,), dtype=state.obs.dtype)
        full_obs = jp.concatenate([state.obs, pad], axis=-1)
        s = self.env.step(state.replace(obs=full_obs), action)
        return s.replace(obs=s.obs[..., :STUDENT_OBS_SIZE])

    @property
    def observation_size(self):
        return STUDENT_OBS_SIZE


# ── Environments ─────────────────────────────────────────────────────────────
# naconmax: GLOBAL contact budget for the Warp broadphase kernel that processes
#           all parallel worlds in one shot. Must be scaled with NUM_ENVS.
# njmax:    per-env constraint budget; do NOT scale with NUM_ENVS.
NACONMAX_TRAIN = 40 * NUM_ENVS
NACONMAX_EVAL  = 40 * NUM_EVAL_ENVS
NJMAX          = 500

raw_env      = registry.load(ENV_NAME, config_overrides={
    "naconmax": NACONMAX_TRAIN,
    "njmax":    NJMAX,
})
raw_eval_env = registry.load(ENV_NAME, config_overrides={
    "naconmax": NACONMAX_EVAL,
    "njmax":    NJMAX,
})

# Same DR pattern as phase1_run: per-device rng split for train (PPO pmaps
# reset), full batch for eval (eval is single-device).
assert NUM_ENVS % NUM_DEVICES == 0, \
    f"NUM_ENVS={NUM_ENVS} not divisible by {NUM_DEVICES} devices"
randomize_train = functools.partial(
    twmr.twmr.domain_randomize_model,
    rng=jax.random.split(jax.random.PRNGKey(SEED), NUM_ENVS // NUM_DEVICES),
)
randomize_eval = functools.partial(
    twmr.twmr.domain_randomize_model,
    rng=jax.random.split(jax.random.PRNGKey(SEED + 1), NUM_EVAL_ENVS),
)

env = StudentObsWrapper(wrapper.wrap_for_brax_training(
    raw_env,
    episode_length=EPISODE_LENGTH,
    action_repeat=1,
    randomization_fn=randomize_train,
))

eval_env = StudentObsWrapper(wrapper.wrap_for_brax_training(
    raw_eval_env,
    episode_length=EPISODE_LENGTH,
    action_repeat=1,
    randomization_fn=randomize_eval,
))

print(f"action_size={raw_env.action_size}  raw_obs_size={raw_env.observation_size}  "
      f"sliced_obs_size={env.observation_size}  num_devices={NUM_DEVICES}")

# Render env: single-env, no DR, also sliced so render_rollout's inference
# pipeline sees 29-dim obs (matches the policy's training-time obs space).
_render_env     = registry.load(ENV_NAME)
_render_wrapped = StudentObsWrapper(wrapper.wrap_for_brax_training(
    _render_env, episode_length=EPISODE_LENGTH, action_repeat=1
))

# ── Action-space constants (from twmr.py) ────────────────────────────────────
_WHEEL_MAX_SPEED = 8.0
_LEG_CENTER      = 1.19
_LEG_HALF_RANGE  = 2.237
_LEG_MIN         = -1.047
_LEG_MAX         =  3.427
_WHEEL_QVEL_IDX  = np.array([6, 10, 14, 18])
_LEG_QPOS_IDX    = np.array([8, 12, 16, 20])


# ── Rollout rendering helper ─────────────────────────────────────────────────
def render_rollout(make_policy, params, outpath, label=""):
    """Run one deterministic episode, save mp4, and plot desired actions."""
    inference_fn = make_policy(params, deterministic=True)
    jit_infer    = jax.jit(inference_fn)

    rng_batch = jax.random.split(jax.random.PRNGKey(SEED + 100), 1)
    state     = jax.jit(_render_wrapped.reset)(rng_batch)

    def step_fn(carry, _):
        s, rng = carry
        rng, k = jax.random.split(rng)
        ks     = jax.random.split(k, 1)
        act    = jax.vmap(jit_infer)(s.obs, ks)[0]
        s      = _render_wrapped.step(s, act)
        return (s, rng), (s.data, act)

    _, (traj, actions) = jax.lax.scan(
        step_fn, (state, jax.random.PRNGKey(SEED)), None, length=EPISODE_LENGTH
    )
    traj.qpos.block_until_ready()

    qpos_np          = np.array(traj.qpos[:, 0])
    qvel_np          = np.array(traj.qvel[:, 0])
    actions_np       = np.array(actions[:, 0])
    ctrl_np          = np.array(traj.ctrl[:, 0])
    wheel_torque_np  = ctrl_np[:, :4]
    actual_wheel_vel = qvel_np[:, _WHEEL_QVEL_IDX]
    actual_leg_pos   = qpos_np[:, _LEG_QPOS_IDX]

    x_positions = qpos_np[:, 0]
    max_x_dist  = float(x_positions.max() - x_positions[0])

    desired_wheel_vel = actions_np[:, :4] * _WHEEL_MAX_SPEED
    desired_leg_pos   = _LEG_CENTER + actions_np[:, 4:] * _LEG_HALF_RANGE
    time_s = np.arange(EPISODE_LENGTH) * 0.02

    outpath_str = str(outpath)
    stem = outpath_str.rsplit(".", 1)[0]

    wheel_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"][:4]

    fig_w, (ax_wv, ax_wt) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    for i in range(4):
        ax_wv.plot(time_s, desired_wheel_vel[:, i], color=wheel_colors[i],
                   linewidth=1.2, label=f"wheel_{i} desired")
        ax_wv.plot(time_s, actual_wheel_vel[:, i], color=wheel_colors[i],
                   linewidth=1.0, linestyle="--", label=f"wheel_{i} actual")
    ax_wv.set_ylabel("Wheel Velocity (rad/s)")
    ax_wv.set_title(f"Wheel Velocity (desired vs actual) — {label}" if label
                    else "Wheel Velocity (desired vs actual)")
    ax_wv.legend(loc="upper right", fontsize=7, ncol=2)
    ax_wv.grid(True, alpha=0.3)

    for i in range(4):
        ax_wt.plot(time_s, wheel_torque_np[:, i], color=wheel_colors[i],
                   linewidth=1.2, label=f"wheel_{i}")
    ax_wt.set_xlabel("Time (s)")
    ax_wt.set_ylabel("Administered Torque (N·m)")
    ax_wt.set_title(f"Administered Wheel Torque — {label}" if label else "Administered Wheel Torque")
    ax_wt.legend(loc="upper right", fontsize=8)
    ax_wt.grid(True, alpha=0.3)

    fig_w.tight_layout()
    wv_path = f"{stem}_wheel_vel.png"
    fig_w.savefig(wv_path, dpi=150)
    plt.show()
    print(f"  Saved: {wv_path}")

    leg_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"][:4]
    fig_l, ax_l = plt.subplots(figsize=(8, 4))
    for i in range(4):
        ax_l.plot(time_s, desired_leg_pos[:, i], color=leg_colors[i],
                  linewidth=1.2, label=f"leg_{i} desired")
        ax_l.plot(time_s, actual_leg_pos[:, i], color=leg_colors[i],
                  linewidth=1.0, linestyle="--", label=f"leg_{i} actual")
    ax_l.axhline(_LEG_MIN, color="grey", ls="--", lw=0.8, label="_LEG_MIN")
    ax_l.axhline(_LEG_CENTER, color="grey", ls="-.", lw=0.8, label="_LEG_CENTER")
    ax_l.axhline(_LEG_MAX, color="grey", ls="--", lw=0.8, label="_LEG_MAX")
    ax_l.set_xlabel("Time (s)")
    ax_l.set_ylabel("Position (rad)")
    ax_l.set_title(f"Leg Position (desired vs actual) — {label}" if label
                   else "Leg Position (desired vs actual)")
    ax_l.legend(loc="upper right", fontsize=7, ncol=2)
    ax_l.grid(True, alpha=0.3)
    fig_l.tight_layout()
    lp_path = f"{stem}_leg_pos.png"
    fig_l.savefig(lp_path, dpi=150)
    plt.show()
    print(f"  Saved: {lp_path}")

    mj_model   = _render_env.mj_model
    renderer   = mujoco.Renderer(mj_model, height=480, width=640)
    cam        = mujoco.MjvCamera()
    mujoco.mjv_defaultFreeCamera(mj_model, cam)
    cam.azimuth, cam.elevation, cam.distance = 135.0, -20.0, 1.5
    chassis_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "chassis")
    d = mujoco.MjData(mj_model)

    scene_option = mujoco.MjvOption()
    mujoco.mjv_defaultOption(scene_option)
    scene_option.flags[mujoco.mjtVisFlag.mjVIS_AUTOCONNECT]  = False
    scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = False
    scene_option.flags[mujoco.mjtVisFlag.mjVIS_PERTFORCE]    = False
    scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT]  = False
    scene_option.geomgroup[3] = 0

    frames = []
    for i in range(qpos_np.shape[0]):
        d.qpos[:] = qpos_np[i]
        d.qvel[:] = qvel_np[i]
        mujoco.mj_forward(mj_model, d)
        cam.lookat[:] = d.xpos[chassis_id]
        renderer.update_scene(d, camera=cam, scene_option=scene_option)
        frames.append(renderer.render())
    renderer.close()

    media.write_video(str(outpath), frames, fps=1.0 / _render_env.dt)
    media.show_video(frames, fps=1.0 / _render_env.dt)
    print(f"  Saved: {outpath}  |  max_dist={max_x_dist:.3f}m")
    return frames


Logs: /home/imeg2025/Transformable-Leg-Wheel-Robot/sandbox/logs/TWMRLegTerr-PPO-20260605-103420
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
action_size=8  raw_obs_size=38  sliced_obs_size=29  num_devices=4


In [4]:
# === DR VERIFICATION ====================================================
# Confirms every domain-randomized parameter actually affects an episode.
# Identical to phase1_run cell 3 (env-level DR is unchanged here — only the
# outermost obs slice is new, and DR runs INSIDE the slice).

from twmr.networks import STUDENT_OBS_SIZE, PRIV_OBS_SIZE
from twmr.twmr import (
    _FRICTION_RANGE, _WHEEL_MOTOR_RANGE, _LEG_MOTOR_RANGE,
    _FLOOR_GEOM_ID,
)

B_VERIFY = NUM_ENVS // NUM_DEVICES
_keys = jax.random.split(jax.random.PRNGKey(0), B_VERIFY)
state0 = jax.jit(env.reset)(_keys)

# After StudentObsWrapper, state.obs is 29-dim; the privileged slice has been
# dropped on the way out but is still present in mjx_model + state.info.
assert state0.obs.shape[-1] == STUDENT_OBS_SIZE, \
    f"expected obs trailing dim {STUDENT_OBS_SIZE}, got {state0.obs.shape}"

def _check_range(name, vals, lo, hi):
    v = np.asarray(vals).ravel()
    ok = (v.min() >= lo - 1e-6) and (v.max() <= hi + 1e-6)
    print(f"  T1 {name:<22s}  [{v.min():+.3f}, {v.max():+.3f}]  "
          f"target [{lo:+.2f}, {hi:+.2f}]  {'PASS' if ok else 'FAIL'}")
    assert ok, f"{name} out of range"

_check_range("friction",             state0.info["friction"],             *_FRICTION_RANGE)
_check_range("wheel_motor_strength", state0.info["wheel_motor_strength"], *_WHEEL_MOTOR_RANGE)
_check_range("leg_motor_strength",   state0.info["leg_motor_strength"],   *_LEG_MOTOR_RANGE)

# Friction is baked into mjx_model.geom_friction; confirm it varies per env
# in the expected range — same logic as phase1's T5b.
_mv = env._mjx_model_v
_geom_fric = np.asarray(_mv.geom_friction[:, _FLOOR_GEOM_ID, 0])
_nom_fric = raw_env._nominal_floor_friction
_fric_scale_from_model = _geom_fric / _nom_fric
in_range = (_fric_scale_from_model.min() >= _FRICTION_RANGE[0] - 1e-6) and \
           (_fric_scale_from_model.max() <= _FRICTION_RANGE[1] + 1e-6)
print(f"  T5b friction (model)        [{_fric_scale_from_model.min():+.3f},"
      f" {_fric_scale_from_model.max():+.3f}]  {'PASS' if in_range else 'FAIL'}")
assert in_range

print("DR VERIFICATION DONE — DR is live; obs is sliced to 29 dims.")
# === END DR VERIFICATION ================================================


Warp CUDA error 2: out of memory (in function wp_cuda_graph_create_exec, /builds/omniverse/warp/warp/native/warp.cu:2899)
E0605 10:34:27.387281 1415525 pjrt_stream_executor_client.cc:2916] Execution of replica 0 failed: UNKNOWN: FFI callback error: RuntimeError: Graph creation error: Warp CUDA error 2: out of memory (in function wp_cuda_graph_create_exec, /builds/omniverse/warp/warp/native/warp.cu:2899)


Traceback (most recent call last):
  File "/home/imeg2025/Transformable-Leg-Wheel-Robot/.venv/lib/python3.11/site-packages/mujoco/mjx/third_party/warp/_src/jax_experimental/ffi.py", line 898, in ffi_callback
    wp.capture_launch(capture.graph)
  File "/home/imeg2025/Transformable-Leg-Wheel-Robot/.venv/lib/python3.11/site-packages/warp/_src/context.py", line 8777, in capture_launch
    raise RuntimeError(f"Graph creation error: {runtime.get_error_string()}")
RuntimeError: Graph creation error: Warp CUDA error 2: out of memory (in function wp_cuda_graph_create_exec, /builds/omniverse/warp/warp/native/warp.cu:2899)



XlaRuntimeError: UNKNOWN: FFI callback error: RuntimeError: Graph creation error: Warp CUDA error 2: out of memory (in function wp_cuda_graph_create_exec, /builds/omniverse/warp/warp/native/warp.cu:2899)

In [ ]:
# ── Network factory ──────────────────────────────────────────────────────────
# Standard Brax PPO MLP (no encoder, no latent z) on the 29-dim student obs.
# Defaults of make_ppo_networks would silently change architecture
# (swish + (32,)*4 / (256,)*5) — pass tanh + (64,64,64) to match phase1.
network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=POLICY_HIDDEN,
    value_hidden_layer_sizes=VALUE_HIDDEN,
    activation=linen.tanh,
)


In [ ]:
times           = [time.monotonic()]
saved_pcts      = set()
_sorted_targets = sorted(TARGET_PCTS)

eval_steps_log:  list[int]   = []
eval_dist_log:   list[float] = []
eval_reward_log: list[float] = []


def progress(num_steps, metrics):
    times.append(time.monotonic())
    reward   = metrics.get("eval/episode_reward", float("nan"))
    max_dist = metrics.get("eval/episode_max_x_dist", float("nan"))
    pct      = num_steps / NUM_TIMESTEPS * 100
    print(f"[{pct:5.1f}%] step={num_steps:>9,}  reward={reward:.4f}  max_dist={max_dist:.3f}m")
    eval_steps_log.append(num_steps)
    eval_dist_log.append(max_dist)
    eval_reward_log.append(reward)


def policy_params_fn(current_step, make_policy, params):
    ckpt_path = ckpt_dir / f"ppo_step_{current_step:09d}"
    model.save_params(str(ckpt_path), params)

    pct = current_step / NUM_TIMESTEPS * 100
    for target in _sorted_targets:
        if pct >= target and target not in saved_pcts:
            saved_pcts.add(target)
            print(f"→ Rendering checkpoint at {pct:.1f}% (target ≥{target}%)…")
            outpath = logdir / f"rollout_{target:03d}pct.mp4"
            render_rollout(make_policy, params, outpath, label=f"{target}%")


train_fn = functools.partial(
    ppo.train,
    num_timesteps         = NUM_TIMESTEPS,
    num_evals             = NUM_EVALS,
    episode_length        = EPISODE_LENGTH,
    num_envs              = NUM_ENVS,
    unroll_length         = UNROLL_LENGTH,
    batch_size            = BATCH_SIZE,
    num_minibatches       = NUM_MINIBATCHES,
    num_updates_per_batch = NUM_UPDATES_PER_BATCH,
    learning_rate         = LEARNING_RATE,
    entropy_cost          = ENTROPY_COST,
    discounting           = DISCOUNTING,
    reward_scaling        = REWARD_SCALING,
    clipping_epsilon      = CLIPPING_EPSILON,
    max_grad_norm         = MAX_GRAD_NORM,
    normalize_observations= True,
    action_repeat         = 1,
    network_factory       = network_factory,
    wrap_env              = False,
)

make_inference_fn, params, _ = train_fn(
    environment      = env,
    eval_env         = eval_env,
    progress_fn      = progress,
    policy_params_fn = policy_params_fn,
)

# Final checkpoint — phase2_run discovers and loads this as the 5th condition.
final_ckpt = ckpt_dir / "ppo_final"
model.save_params(str(final_ckpt), params)
print(f"\nSaved final checkpoint: {final_ckpt}")

print(f"\nJIT compile time : {times[1] - times[0]:.1f}s")
print(f"Total train time : {times[-1] - times[1]:.1f}s")


In [ ]:
# ── Plot: max forward distance vs training timesteps ─────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.array(eval_steps_log)/1e6, eval_dist_log, marker="o", markersize=4, linewidth=1.5)
ax.set_xlabel("Training Timesteps (10^6)")
ax.set_ylabel("Max Forward Distance (m)")
ax.set_title("Max +x Distance Reached per Episode During Training")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(str(logdir / "max_x_dist_vs_steps.png"), dpi=150)
plt.show()
print(f"  Plot saved: {logdir / 'max_x_dist_vs_steps.png'}")

print("Rendering final rollout…")
frames = render_rollout(make_inference_fn, params, logdir / "rollout_final.mp4", label="final")
media.show_video(frames, fps=1.0 / _render_env.dt)


In [ ]:
# ── M1 verification: checkpoint round-trip ────────────────────────────────────
# Load ppo_final, rebuild the inference function, run one deterministic
# episode in the (un-randomized) render env, and compare to the last training
# eval reward. Pass: within ±25% (single-env vs 128-env mean).
loaded_params = model.load_params(str(final_ckpt))
print(f"Loaded params from: {final_ckpt}")

network = network_factory(
    observation_size=STUDENT_OBS_SIZE,            # 29-dim student obs
    action_size=raw_env.action_size,              # 8
    preprocess_observations_fn=running_statistics.normalize,
)
make_inference_fn_loaded = ppo_networks.make_inference_fn(network)
inference_fn_loaded = make_inference_fn_loaded(loaded_params, deterministic=True)
jit_infer = jax.jit(inference_fn_loaded)

rng_batch = jax.random.split(jax.random.PRNGKey(SEED + 100), 1)
state = jax.jit(_render_wrapped.reset)(rng_batch)

def _step_loaded(carry, _):
    s, rng = carry
    rng, k = jax.random.split(rng)
    ks = jax.random.split(k, 1)
    act = jax.vmap(jit_infer)(s.obs, ks)[0]
    s = _render_wrapped.step(s, act)
    return (s, rng), s.reward

(_, _), rewards = jax.lax.scan(
    _step_loaded, (state, jax.random.PRNGKey(SEED)), None, length=EPISODE_LENGTH
)
ep_reward_loaded = float(jp.sum(rewards))
last_eval_reward = eval_reward_log[-1] if eval_reward_log else float("nan")

print(f"Loaded-checkpoint episode reward (1 env, deterministic): {ep_reward_loaded:.4f}")
print(f"Last training-eval reward (mean over {NUM_EVAL_ENVS} envs):   {last_eval_reward:.4f}")
if last_eval_reward == last_eval_reward:
    rel = (ep_reward_loaded - last_eval_reward) / abs(last_eval_reward)
    print(f"Relative gap: {rel:+.1%}")
    assert abs(rel) < 0.25, f"Round-trip reward gap too large: {rel:+.1%}"
    print("M1 PASS — checkpoint round-trip reproduces policy behavior.")
